Pydantic Monty runs Python a model wrote inside a sandbox with no file, network or environment access. This distribution is a metapackage: it installs NO Python module of its own, only the `pydantic-monty` command, and pins the two halves to its own version: `pydantic_monty_client` (the module `pydantic_monty`) and `pydantic_monty_runtime` (the worker program `monty`). The first cell checks what this distribution really owns (the pins and the command), and the examples run the two halves together. Needs no network or GPU.

In [ ]:
import importlib.metadata
import shutil
import subprocess

distribution = importlib.metadata.distribution("pydantic_monty")
owned = sorted(str(path) for path in distribution.files if ".dist-info" not in str(path))
assert all("pydantic-monty" in path for path in owned) and len(owned) == 1, owned   # the command, and no module
halves = {name: importlib.metadata.version(name) for name in ("pydantic_monty_client", "pydantic_monty_runtime")}
assert set(halves.values()) == {distribution.version}, halves
scripts = [entry for entry in distribution.entry_points if entry.group == "console_scripts"]
assert [(entry.name, entry.value) for entry in scripts] == [("pydantic-monty", "pydantic_monty._cli:main")], scripts
command = shutil.which("pydantic-monty")
assert command is not None, "the `pydantic-monty` command is not on PATH"
shown = subprocess.run([command, "--version"], capture_output=True, text=True, timeout=120)
assert shown.returncode == 0 and shown.stdout.strip() == f"monty-runtime {distribution.version}", (shown.returncode, shown.stdout, shown.stderr)
print("pydantic_monty", distribution.version, "owns", owned, "; halves:", halves, "; `pydantic-monty --version` ->", shown.stdout.strip())

## Example 1 — the two halves find each other: the module starts the worker program and gets an answer

In [ ]:
import os
import pydantic_monty
from pydantic_monty._binary import find_monty_binary

worker = find_monty_binary()
assert os.path.isfile(worker) and os.access(worker, os.X_OK), worker
with pydantic_monty.Monty() as pool, pool.checkout() as session:
    assert session.feed_run("sum(range(101))") == 5050
    assert isinstance(session.worker_pid, int) and session.worker_pid != os.getpid()
print("pydantic_monty", pydantic_monty.__version__, "ran a snippet in a separate worker process:", worker)

## Example 2 — "code mode": one sandboxed program calls two host tools and combines the answers

In [ ]:
from pydantic_monty import Monty

log = []
def get_weather(city):
    log.append(("get_weather", city))
    return {"Paris": 21, "Oslo": 9}[city]
def convert(celsius):
    log.append(("convert", celsius))
    return celsius * 9 / 5 + 32
program = '''
readings = {}
for city in cities:
    readings[city] = convert(get_weather(city))
warmest = max(readings, key=lambda city: readings[city])
{"warmest": warmest, "fahrenheit": readings}
'''
with Monty() as pool, pool.checkout() as session:
    result = session.feed_run(program, inputs={"cities": ["Paris", "Oslo"]},
                              external_lookup={"get_weather": get_weather, "convert": convert})
assert result == {"warmest": "Paris", "fahrenheit": {"Paris": 69.8, "Oslo": 48.2}}, result
assert [name for name, _ in log] == ["get_weather", "convert", "get_weather", "convert"], log
print("one program, four host calls:", result)

## Example 3 — values cross the boundary intact, and print output is captured

In [ ]:
import datetime
from pydantic_monty import CollectString, Monty

values = {"n": 2 ** 70, "ratio": 0.1, "text": "naïve 日本語", "raw": b"\x00\xff", "items": [1, (2, 3), {"k": None}],
          "when": datetime.date(2026, 10, 2), "flags": {True, False}}
printed = CollectString()
with Monty() as pool, pool.checkout() as session:
    echoed = session.feed_run("print(len(values), sorted(values))\nvalues", inputs={"values": values}, print_callback=printed)
assert echoed == values, echoed
assert printed.output.startswith("7 ['flags', 'items', 'n', 'ratio', 'raw', 'text', 'when']"), printed.output
print("7 values of 7 types went into the sandbox and came back equal; the sandbox printed:", printed.output.strip())

## Example 4 — type checking before running, and errors with a traceback

In [ ]:
from pydantic_monty import Monty, MontyRuntimeError, MontyTypingError

with Monty() as pool:
    with pool.checkout(type_check=True) as session:
        try:
            session.feed_run('count: int = "three"')
            raise AssertionError("a type error was not reported")
        except MontyTypingError as error:
            first_line = error.display().splitlines()[0]
            assert "invalid-assignment" in first_line, first_line
        assert session.feed_run("count: int = 3\ncount + 1") == 4
    with pool.checkout() as session:
        try:
            session.feed_run("def divide(a, b):\n    return a / b\ndivide(1, 0)")
            raise AssertionError("division by zero did not fail")
        except MontyRuntimeError as error:
            frames = error.traceback()
            assert isinstance(error.exception(), ZeroDivisionError) and frames[-1].function_name == "divide", frames
            assert frames[-1].line == 2 and frames[-1].source_line.strip() == "return a / b", frames[-1].dict()
print("type checker:", first_line, "| runtime error traced to line 2 of divide()")

## Example 5 — the sandbox walls: no host files, no environment, time and memory limits

In [ ]:
from pydantic_monty import Monty, MontyRuntimeError

refused = {}
with Monty() as pool:
    with pool.checkout() as session:
        for label, code in (("host file", 'open("/etc/hostname").read()'), ("environment", 'import os\nos.environ["HOME"]')):
            try:
                session.feed_run(code)
                raise AssertionError(f"the sandbox allowed {code!r}")
            except MontyRuntimeError as error:
                refused[label] = type(error.exception()).__name__
    for label, limits, code in (("endless loop", {"max_feed_duration_secs": 0.2}, "while True:\n    pass"),
                                ("memory", {"max_memory": 1_000_000}, "blob = [0] * 50_000_000"),
                                ("recursion", {"max_recursion_depth": 50}, "def down(n):\n    return down(n + 1)\ndown(0)")):
        with pool.checkout(limits=limits) as session:
            try:
                session.feed_run(code)
                raise AssertionError(f"{label}: the limit did not stop the code")
            except MontyRuntimeError as error:
                refused[label] = type(error.exception()).__name__
assert refused == {"host file": "PermissionError", "environment": "RuntimeError", "endless loop": "TimeoutError",
                   "memory": "MemoryError", "recursion": "RecursionError"}, refused
print("refused or stopped:", refused)